# F-09 Whisper 3차 학습 — 재전처리 + final2 이어받기

**변경 사항 (finetune.ipynb 대비)**
- 전처리: `preprocess_v2.py` 사용 — SP/FP 태그 처리 추가
- 데이터: `processed/senior_speech_v2` (기존 v1 보존)
- 학습: `final2` LoRA 가중치 이어받아 3차 학습 (lr=1e-5, 2000 steps)
- 평가: `suppress_tokens` 기본값 유지 + 구두점 제거 CER

**실행 순서**: 셀 01 → 02 → 03 → 04

In [ ]:
# 셀 01 — 라이브러리 설치 + Drive 마운트 + 경로 설정
!pip install -q \
    transformers \
    datasets \
    peft \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    tensorboard

from google.colab import drive
from pathlib import Path
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT      = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH    = DRIVE_ROOT / 'processed/senior_speech_v2'   # v2 데이터셋
CHECKPOINT_DIR  = DRIVE_ROOT / 'checkpoints/whisper-senior'
FINAL2_DIR      = CHECKPOINT_DIR / 'final2'                   # 2차 학습 결과 (이어받기 시작점)
FINAL3_DIR      = CHECKPOINT_DIR / 'final3'                   # 3차 학습 결과
CHECKPOINT3_DIR = CHECKPOINT_DIR / 'stage3'
CHECKPOINT3_DIR.mkdir(parents=True, exist_ok=True)

print('완료')
print(f'데이터셋 경로 : {DATASET_PATH}')
print(f'체크포인트 경로: {CHECKPOINT_DIR}')

In [ ]:
# 셀 02 — 재전처리 (senior_speech_v2 없을 때만 실행)
import shutil

if not DATASET_PATH.exists():
    print('전처리 시작 — preprocess_v2.py 실행')
    shutil.copy('/content/drive/MyDrive/Dadam/whisper/preprocess_v2.py', '/content/preprocess_v2.py')
    !python /content/preprocess_v2.py
    print('전처리 완료')
else:
    print(f'v2 데이터셋 이미 존재 — 건너뜀: {DATASET_PATH}')

In [ ]:
# 셀 03 — 3차 학습
# 백그라운드 실행 활성화 후 실행할 것 (런타임 > 백그라운드 실행)
import os
import re
import torch
import evaluate
from dataclasses import dataclass
from typing import Any
from datasets import load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import PeftModel

MODEL_ID     = 'openai/whisper-large-v3-turbo'
EVAL_SAMPLES = 500

if not FINAL2_DIR.exists():
    raise FileNotFoundError(f'2차 학습 결과 없음: {FINAL2_DIR}')

processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')

# float32 로드 — Trainer fp16=True가 자동 변환
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
base_model.generation_config.language           = 'korean'
base_model.generation_config.task               = 'transcribe'
base_model.generation_config.forced_decoder_ids = None
# suppress_tokens 건드리지 않음 — Whisper 기본값 유지 (repetition 방지)

# is_trainable=True — 3차 학습을 위해 LoRA 파라미터 그래디언트 활성화
model = PeftModel.from_pretrained(base_model, str(FINAL2_DIR), is_trainable=True)
model.print_trainable_parameters()

dataset = load_from_disk(str(DATASET_PATH))
print(dataset)

# ── 텍스트 정규화 (평가용) ───────────────────────────────────────────────────
PUNCT_PATTERN = re.compile(r'[.?!,。、]')

def clean_text(text: str) -> str:
    """구두점·공백 제거 — 음절 단위 CER 계산용"""
    return PUNCT_PATTERN.sub('', text).replace(' ', '').strip()

# ── DataCollator ─────────────────────────────────────────────────────────────
@dataclass
class WhisperDataCollator:
    """배치별 패딩 처리 — 가변 길이 오디오를 30초로 패딩"""
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        labels = self.processor.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=448,
        ).input_ids
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'input_features': inputs.input_features, 'labels': labels}

# ── CER 메트릭 ───────────────────────────────────────────────────────────────
cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    # 구두점·공백 제거 후 음절 단위 CER — v1에서 구두점 불일치로 수치가 부풀었던 문제 해결
    pred_str  = [clean_text(p) for p in pred_str]
    label_str = [clean_text(l) for l in label_str]
    return {'cer': round(cer_metric.compute(predictions=pred_str, references=label_str), 4)}

# ── 학습 설정 ─────────────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT3_DIR),
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,         # 유효 배치: 64
    learning_rate=1e-5,                    # final2 이어받기 — 낮은 lr로 미세 조정
    warmup_steps=100,
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    report_to='tensorboard',
    save_total_limit=3,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'].select(range(EVAL_SAMPLES)),
    data_collator=WhisperDataCollator(processor=processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

# 체크포인트 이어받기
last_checkpoint = None
checkpoints = sorted(CHECKPOINT3_DIR.glob('checkpoint-*'), key=os.path.getmtime)
if checkpoints:
    last_checkpoint = str(checkpoints[-1])
    print(f'체크포인트 발견 — 이어서 학습: {last_checkpoint}')
else:
    print('체크포인트 없음 — 처음부터 3차 학습 시작')

trainer.train(resume_from_checkpoint=last_checkpoint)

model.save_pretrained(str(FINAL3_DIR))
processor.save_pretrained(str(FINAL3_DIR))
print(f'3차 학습 완료. 저장 위치: {FINAL3_DIR}')

In [ ]:
# 셀 04 — final3 평가 (구두점+공백 제거 CER)
# 셀 03 완료 후 실행 — model/processor/dataset/clean_text/GEN_KWARGS 재사용
import evaluate
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any

cer_metric   = evaluate.load('cer')
EVAL_SAMPLES = 500

# suppress_tokens 기본값 유지 — repetition 방지
GEN_KWARGS = dict(language='korean', task='transcribe', num_beams=1)

# 평가용 모델 로드 (셀 03을 실행하지 않은 경우)
if 'model' not in dir() or not hasattr(model, 'generate'):
    import torch
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    from peft import PeftModel
    MODEL_ID   = 'openai/whisper-large-v3-turbo'
    processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')
    base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    base_model.generation_config.language           = 'korean'
    base_model.generation_config.task               = 'transcribe'
    base_model.generation_config.forced_decoder_ids = None
    model = PeftModel.from_pretrained(base_model, str(FINAL3_DIR), is_trainable=False)
    model = model.to('cuda').eval()
    from datasets import load_from_disk
    dataset = load_from_disk(str(DATASET_PATH))

@dataclass
class EvalCollator:
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        return {'input_features': inputs.input_features, 'texts': texts}

eval_subset = dataset['validation'].select(range(EVAL_SAMPLES))
loader      = DataLoader(eval_subset, batch_size=16, collate_fn=EvalCollator(processor))

all_preds, all_refs = [], []

for batch in loader:
    input_feats = batch['input_features'].to('cuda').half()
    with torch.no_grad():
        pred_ids = model.generate(input_features=input_feats, **GEN_KWARGS)
    preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
    all_preds.extend([clean_text(p) for p in preds])
    all_refs.extend( [clean_text(r) for r in batch['texts']])

cer_result = cer_metric.compute(predictions=all_preds, references=all_refs)

print(f'final3 CER ({EVAL_SAMPLES}개): {cer_result:.4f}  →  {cer_result * 100:.2f}%')
print()
print('── 예측 vs 정답 샘플 5개 ──')
for i in range(5):
    print(f'  정답: {all_refs[i]}')
    print(f'  예측: {all_preds[i]}')
    print()